# Chapter 3 - Lab 7: <font color='blue'>Claude SDK (Anthropic)</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using Anthropic's Claude SDK** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

This lab is deliberately different from the previous six. Instead of building an agent that calls tools, you will make a **direct LLM call** to Claude — embedding the finance data in the prompt and asking the model to do the math. This is the simplest possible 'agent', and it gives you a baseline to compare against the tool-using frameworks.

Working through this lab will sharpen your intuition for when tool calling pays off and when a strong base model with a good prompt is enough.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **Anthropic SDK** (`anthropic`) — direct `client.messages.create` API.
* **Claude Sonnet 3.5** — capable enough to compute P/E ratios in-context with no tools.
* **Prompt-only data passing** — no function calling, no tools, no agent loop.

You will need an OpenAI API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [ ]:
%pip install -q anthropic pydantic python-dotenv

## 2. Set up the API key(s)

If you are running this notebook in **Google Colab**, add your OpenAI key in the left vertical menu (the *key* icon) under the name `OPENAI_API_KEY`.

If you are running locally, replace the cell below with `os.environ['OPENAI_API_KEY'] = '...'` or load it from a `.env` file.

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

## 4. Build a tools-free prompt

We embed the finance data into the user prompt and ask Claude to do the work in one shot. No tool registration, no agent loop. This is what the chapter calls the *non-agentic* baseline.

In [ ]:
enhanced_prompt = f"""
{input_message}

Available stock data:
- AAPL: Price=$195.3, EPS=$6.67
- JPM:  Price=$148.7, EPS=$12.61

Please:
1. Calculate P/E ratios for both stocks (P/E = Price / EPS)
2. Compare the valuation metrics
3. Write a brief investment memo
"""

## 5. Call Claude

We use Anthropic's `messages.create` directly. The `system` parameter takes our shared system message; `max_tokens` caps the reply length.

In [ ]:
import os
from anthropic import Anthropic

client = Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])

response = client.messages.create(
    model='claude-3-5-sonnet-20241022',
    max_tokens=1000,
    temperature=0,
    system=system_message,
    messages=[{'role': 'user', 'content': enhanced_prompt}],
)

## 6. Read the response

In [ ]:
for block in response.content:
    if block.type == 'text':
        print(block.text)

## 7. Results

You should see the agent call `get_stock_data` twice (once per ticker), then call `compute_pe` twice to compute each P/E ratio, and finally produce a short memo comparing the two.

**What to notice about Anthropic's Claude SDK (no tools) specifically:**

* When the dataset is small enough to fit in the prompt and the math is simple, **no agent is needed**. A strong base model and a well-structured prompt are enough.
* This baseline matters: every framework you have seen so far adds cost (more tokens, more latency, more code) — so they must earn that cost with capabilities a direct call cannot provide.
* Capabilities that *do* justify the agent overhead: dynamic data sources, multi-step planning, fact validation, multi-agent collaboration, and reproducible audit trails.
* In Lab 8 you will see how to do tool calling natively with Claude (via the Anthropic API's `tools` parameter).